# 03 — Minimum-Variance Hedge Ratio and Validation
Business question: what hedge ratio historically minimized residual price variance, and does that estimate remain useful out of sample?


## Reconstruct the monthly market frame from the saved analysis

In [ ]:
import pandas as pd

from src.min_variance import minimum_variance_hedge_ratio
from src.validation import (
    bootstrap_minimum_variance_ratio,
    summarize_walk_forward,
    walk_forward_min_variance,
)

monthly = pd.read_csv(
    "../outputs/monthly_analysis.csv",
    parse_dates=["month"],
).set_index("month")

market = monthly[
    ["spot_exit", "futures_entry", "futures_exit"]
]

ratio = minimum_variance_hedge_ratio(market)
ratio

## Walk-forward test

Each test observation uses only the trailing historical window to estimate the hedge ratio.

In [ ]:
walk = walk_forward_min_variance(market, window=24)
pd.DataFrame([summarize_walk_forward(walk)])

## Bootstrap uncertainty

The confidence interval shows that the point estimate should not be treated as an exact constant.

In [ ]:
bootstrap_summary, bootstrap_samples = (
    bootstrap_minimum_variance_ratio(
        market,
        n_bootstrap=5_000,
        confidence=0.95,
        seed=42,
    )
)
bootstrap_summary

## Interpretation

The statistically estimated ratio is a benchmark. A commercial hedge can still be capped below the estimate because of production uncertainty, internal policy, or other residual risks.